In [1]:
#Apply PCA
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import Birch
from sklearn.cluster import OPTICS
from sklearn.decomposition import PCA
import numpy as np
import time
# Load dataset
df= pd.read_csv("data/data.csv")

In [2]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])
df.isnull().sum().sum()
df = df.dropna(axis=1, how='all')
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,842517,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,84300903,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,84348301,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,84358402,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,1,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,926682,1,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,926954,1,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,927241,1,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [3]:

# Drop target and ID columns
X = df.drop(columns=["id"], errors="ignore")
print("Features shape:", X.shape)

#apply scaling
scaler = StandardScaler()
X_sp = scaler.fit_transform(X)

print("Features shape (scaled version):", X_sp.shape)



Features shape: (569, 31)
Features shape (scaled version): (569, 31)


In [4]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_sp)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_pca.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (569, 11)
Explained variance ratio sum: 0.9547787315763366


In [5]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [6]:
#K-Means on Scaled + PCA Data
start_time = time.time()
kmean_pca = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    kmean_pca.append({"algorithm": "KMeans", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")   


Runtime: 5.419275999069214 seconds
K-Means runtime: 5.4193 seconds


In [7]:
#Gaussian Mixture (GMM)on Scaled + PCA Data
start_time = time.time()
gmm_pca = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    gmm_pca.append({"algorithm": "GMM", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 2.9125540256500244 seconds
GMM runtime: 2.9126 seconds


In [8]:
#Agglomerative Clustering on Scaled + PCA Data
start_time = time.time()
agg_pca = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   

Runtime: 0.16221141815185547 seconds
Agglomerative runtime: 0.1622 seconds


In [9]:
#Spectral Clustering on Scaled + PCA Data
start_time = time.time()
spec_pca = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    spec_pca.append({"algorithm": "Spectral", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 0.5914835929870605 seconds
Spectral runtime: 0.5915 seconds


In [10]:
#DBSCAN on Scaled + PCA Data
start_time = time.time()
dbscan_pca = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_pca[mask], labels[mask])
        dbscan_pca.append({"algorithm": "DBSCAN", "preprocessing": "PCA", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 0.04331064224243164 seconds
DBSCAN runtime: 0.0433 seconds


In [11]:
#OPTICS on Scaled + PCA Data
start_time = time.time()
optics_pca = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_pca)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        optics_pca.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 59.56792449951172 seconds
OPTICS runtime: 59.5679 seconds


In [12]:
#BIRCH on Scaled + PCA Data
start_time = time.time()
birch_pca = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_pca)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_pca) and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        birch_pca.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 0.4123857021331787 seconds
BIRCH runtime: 0.4124 seconds


In [13]:
import csv

breast_cancer_results_pca = (kmean_pca + gmm_pca + agg_pca + spec_pca + dbscan_pca+birch_pca+optics_pca)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/breast_cancer_data/breast_cancer_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(breast_cancer_results_pca)


In [13]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis 
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_pca:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_pca:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_pca:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_pca:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_pca:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_pca:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_pca:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# --- helper function to fit a model and return labels ---
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_pca)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_pca), size=len(X_pca), replace=True)
        X_boot = X_pca[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\n BOOTSTRAP ARI STABILITY ")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\metrics\cluster\_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\metrics\cluster\_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\metrics\cluster\_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = typ


 BOOTSTRAP ARI STABILITY 
    algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples
      K-Means 2.0    0.9593   0.0450  NaN        NaN          NaN
      K-Means 3.0    0.8855   0.0897  NaN        NaN          NaN
      K-Means 4.0    0.7422   0.1867  NaN        NaN          NaN
      K-Means 5.0    0.6616   0.1455  NaN        NaN          NaN
      K-Means 6.0    0.7018   0.1410  NaN        NaN          NaN
      K-Means 7.0    0.6182   0.1288  NaN        NaN          NaN
      K-Means 8.0    0.5863   0.1134  NaN        NaN          NaN
          GMM 2.0    0.9437   0.0547  NaN        NaN          NaN
          GMM 3.0    0.8235   0.0760  NaN        NaN          NaN
          GMM 4.0    0.7140   0.0940  NaN        NaN          NaN
          GMM 5.0    0.6271   0.0874  NaN        NaN          NaN
          GMM 6.0    0.6009   0.0789  NaN        NaN          NaN
          GMM 7.0    0.5606   0.0936  NaN        NaN          NaN
          GMM 8.0    0.5640   0.0932  NaN        

In [14]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  Stability Score
    BIRCH NaN    0.9916   0.0068           0.9864
    BIRCH NaN    0.6918   0.0324           0.9352
 Spectral 2.0    0.9359   0.0337           0.9326


In [15]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples  Stability Score
    BIRCH NaN    0.9916   0.0068  NaN        0.5          NaN           0.9864
  K-Means 2.0    0.9593   0.0450  NaN        NaN          NaN           0.9100
      GMM 2.0    0.9437   0.0547  NaN        NaN          NaN           0.8906


In [19]:
ari_df.to_csv("updated_data/ARI_Score/cancer_pca_ari.csv", index=False)

In [17]:
#  Combine all algorithm results 
all_results = (
    kmean_pca +
    gmm_pca +
    agg_pca +
    spec_pca +
    dbscan_pca +
    birch_pca +
    optics_pca
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\n TOP 3 SILHOUETTE")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\n TOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better)
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\nTOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

#  Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

# Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse)
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


 TOP 3 SILHOUETTE
    algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
       KMeans 2.0  NaN        NaN          NaN         NaN      0.3690
Agglomerative 2.0  NaN        NaN          NaN         NaN      0.3445
     Spectral 2.0  NaN        NaN          NaN         NaN      0.3409

 TOP 3 DAVIES-BOULDIN 
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
    BIRCH NaN  NaN        0.5          NaN       561.0          0.1557
    BIRCH NaN  NaN        1.0          NaN       422.0          0.5057
    BIRCH NaN  NaN        1.5          NaN       270.0          0.7684

TOP 3 CALINSKI-HARABASZ 
    algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
       KMeans 2.0  NaN        NaN          NaN         NaN           303.2883
Agglomerative 2.0  NaN        NaN          NaN         NaN           281.6127
     Spectral 2.0  NaN        NaN          NaN         NaN           281.1529

 BOTTOM 3 SILHOUETTE 
algorithm   k  eps  threshol

In [17]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_pca,
    "GMM": gmm_pca,
    "Agglomerative": agg_pca,
    "Spectral": spec_pca,
    "DBSCAN": dbscan_pca,
    "BIRCH": birch_pca,
    "OPTICS": optics_pca
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    print(algorithm)
   



   
    # Select parameter column
   

    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



    # TOP 3 SILHOUETTE

    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



    
    # TOP 3 DAVIES-BOULDIN
  

    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



 
    # TOP 3 CALINSKI-HARABASZ
  

    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.3690
 3      0.3312
 4      0.2865

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.2197
 4          1.4079
 3          1.4915

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           303.2883
 3           219.4329
 4           179.0017


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.3258
 3      0.2973
 4      0.2966

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.3266
 4          1.6337
 3          1.8567

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           264.9731
 3           158.9774
 4           122.1793


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.3445
 3      0.3232
 5      0.2959

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.2935
 5          1.4509
 4          1.5421

Top 3 Calinski-H